# FILIP GUARD — continuarea evaluării calității AI

Proba reală `craft` a confirmat deja HTTP 200, 232 tokenuri și 1,258 s. Nu o repetăm.

Urmează **cel mult nouă cazuri sintetice noi**, în engleză și română: cele șase rămase, plus H1, legendă de imagine și conținut insuficient. Nu sunt scanate site-uri reale și nu se execută reparații. Unele exemple au dovezi concurente; acceptarea JSON-ului este măsurată separat de relevanță.

Păstrează `GROQ_API_KEY` existent în Secrets și runtime CPU gratuit. Confirmă planul Free, transferul exemplelor sintetice și cele maximum nouă apeluri, apoi rulează celulele. Se păstrează aceeași bază de date de buget și toate perioadele de așteptare. Cererile sunt la minimum 61 s distanță, fără retry. O limită activă sau o eroare decisivă oprește restul cazurilor.

Nu adăuga un card, nu schimba cheia și nu reseta sesiunea/baza de date pentru a ocoli limite. Dacă se oprește din cauza cotei, păstrează raportul. La final descarcă JSON-ul fără secrete. Evaluarea umană a relevanței și calitatea completă rămân distincte de verificările automate.

Reluarea reutilizează numai rezultatele acceptate pentru exact aceleași exemple sintetice, model și versiune de contract, inclusiv după expirarea cache-ului. În sesiunea cu cele trei rezultate existente rămân maximum șase apeluri noi. Cheia este citită o singură dată la configurare și păstrată exclusiv în memoria runtime-ului. Nu se resetează bugetul sau cooldown-ul. Dacă runtime-ul gratuit a fost șters de Colab, istoricul local nu poate fi recuperat automat; nu interpreta absența sa ca resetarea cotei Groq.


In [ ]:
import os, pathlib, subprocess, sys, re, sqlite3
os.umask(0o077)
repository='https://github.com/Ciobanu87/filip-guard.git'
branch='filip-guard-development'
remote=subprocess.check_output(['git','ls-remote',repository,'refs/heads/'+branch],text=True).split()[0]
if not re.fullmatch('[0-9a-f]{40}',remote):raise RuntimeError('Development revision could not be verified')
checkout=pathlib.Path('/content/filip-guard-ai-validation-'+remote[:12])
if not checkout.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',branch,repository,str(checkout)],check=True,capture_output=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
if revision!=remote:raise RuntimeError('Development branch changed during setup; rerun this setup cell')
print('Repository development revision:',revision)
subprocess.run([sys.executable,'-m','pip','install','-q','beautifulsoup4>=4.12,<5','cryptography>=44,<49','requests>=2.32,<3'],check=True)
# A notebook kernel can retain old Python modules even after a new clone.
for name,module in list(sys.modules.items()):
    origin=getattr(module,'__file__',None)
    if isinstance(origin,str) and origin.startswith('/content/filip-guard-ai-validation'):
        del sys.modules[name]
sys.path[:]=[entry for entry in sys.path if not str(entry).startswith('/content/filip-guard-ai-validation')]
sys.path.insert(0,str(checkout))
# Share the test budget across source revisions; do not erase old reservations.
state=pathlib.Path('/content/filip-guard-cloud-validation-state');state.mkdir(exist_ok=True,mode=0o700)
ledger=state/'ledger.sqlite'
legacy=pathlib.Path('/content/filip-guard-ai-validation/validation-ledger.sqlite')
if not ledger.exists() and legacy.is_file():
    source=sqlite3.connect(legacy.resolve().as_uri()+'?mode=ro',uri=True)
    target=sqlite3.connect(ledger)
    try:source.backup(target)
    finally:source.close();target.close()
    ledger.chmod(0o600)


In [ ]:
advisor=None  # Failed setup must not leave an older provider usable.
_previous_credential=globals().get('credential')
if hasattr(_previous_credential,'close'):_previous_credential.close()
credential=None
#@title Confirmări obligatorii — nu activa plăți
free_organization_without_billing = False #@param {type:"boolean"}
synthetic_data_transfer_authorized = False #@param {type:"boolean"}
bounded_quality_requests_authorized = False #@param {type:"boolean"}
if not all((free_organization_without_billing,synthetic_data_transfer_authorized,bounded_quality_requests_authorized)):
    raise RuntimeError('Confirmă organizația Free fără facturare și testele sintetice înainte de rulare.')
from google.colab import userdata
from advisor_groq import GroqFreeProvider
from contextual_advisor import ContextualAdvisor
from advisor_quality_evaluation import run_quality_evaluation
from advisor_credentials import SessionCredential
# One UI secret read in this setup cell. No later userdata.get() during evaluation.
credential=SessionCredential.load(lambda:userdata.get('GROQ_API_KEY'))
provider=GroqFreeProvider(credential,free_plan_confirmed=True,data_consent=True)
advisor=ContextualAdvisor(ledger,provider,daily_calls=20,timeout=20,max_output_tokens=256)
print('Configurație pregătită. Nicio inferență nu a fost executată de această celulă.')


In [ ]:
if advisor is None:raise RuntimeError('Configurarea nu a reușit; evaluarea rămâne blocată.')
import json
report=run_quality_evaluation(advisor,live=True,progress=lambda result:print(json.dumps(result,ensure_ascii=False),flush=True))
report['repository_revision']=revision
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Inferență reală nouă:',report['real_inference_executed'])
print('Rezultate AI acceptate:',report['accepted_ai_cases'])
print('Calitatea necesită evaluare umană:',not report['quality_validated'])
print('Cazuri care trec verificările observabile:',report['observable_quality_passes'])
print('Aceste verificări nu înlocuiesc evaluarea semantică umană.')
print('Apeluri furnizor încercate:',report['attempted_cases'])
print('Cazuri acceptate anterior, nerepetate:',report['completed_before_this_run'])
print('Maximum cazuri rămase:',report['remaining_provider_cases_max'])


In [ ]:
import uuid
from google.colab import files
report_path=checkout/('cloud-validation-'+uuid.uuid4().hex+'.json')
with report_path.open('x',encoding='utf-8') as out:
    json.dump(report,out,ensure_ascii=False,indent=2)
report_path.chmod(0o600)
files.download(str(report_path))
